# AMA Supercross Data Exploration

Exploring the scraped race data from 2023-2025 seasons to understand:
- Data quality and completeness
- Rider performance patterns
- Feature engineering opportunities
- Model building strategy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load data
data_path = Path('../data/raw/sx_all_results.csv')
df = pd.read_csv(data_path)

print(f"Total records: {len(df):,}")
print(f"\nColumns: {df.columns.tolist()}")
df.head(10)

## 1. Data Overview

In [ ]:
# Basic statistics
print("=" * 70)
print("DATA SUMMARY")
print("=" * 70)
print(f"\nYears: {sorted(df['year'].unique())}")
print(f"Classes: {sorted(df['class'].unique())}")
print(f"Event types: {sorted(df['event_type'].unique())}")
print(f"\nTotal races: {df['race'].nunique()}")
print(f"Total riders: {df['rider'].nunique()}")
print(f"Total results: {len(df):,}")

print("\n" + "=" * 70)
print("RESULTS BY YEAR AND CLASS")
print("=" * 70)
print(df.groupby(['year', 'class']).size().unstack(fill_value=0))

print("\n" + "=" * 70)
print("RESULTS BY EVENT TYPE")
print("=" * 70)
print(df['event_type'].value_counts())

In [ ]:
# Check for missing data
print("Missing data:")
print(df.isnull().sum())

print("\nMissing dates by year:")
print(df[df['date'].isnull()].groupby('year')['race'].value_counts())

## 2. Main Event Analysis

Focus on main events since that's what we want to predict

In [ ]:
# Filter to main events only
main_events = df[df['event_type'] == 'main_event'].copy()

print(f"Main event results: {len(main_events):,}")
print(f"Races with main events: {main_events['race'].nunique()}")
print(f"Riders in main events: {main_events['rider'].nunique()}")

print("\nMain events by class:")
print(main_events.groupby(['year', 'class']).size().unstack(fill_value=0))

In [ ]:
# Average field size per race
field_sizes = main_events.groupby(['year', 'class', 'race']).size().reset_index(name='field_size')

print("Average field size by class:")
print(field_sizes.groupby('class')['field_size'].describe())

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
field_sizes.boxplot(column='field_size', by='class', ax=ax)
plt.suptitle('')
plt.title('Main Event Field Sizes by Class')
plt.ylabel('Number of Riders')
plt.tight_layout()
plt.show()

## 3. Rider Performance Analysis

In [ ]:
# Top riders by wins (main events only)
wins = main_events[main_events['position'] == 1].groupby('rider').size().sort_values(ascending=False)

print("Top 15 riders by main event wins (2023-2025):")
print(wins.head(15))

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))
wins.head(15).plot(kind='barh', ax=ax)
plt.title('Top 15 Riders by Main Event Wins (2023-2025)')
plt.xlabel('Number of Wins')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Top riders by podiums (top 3)
podiums = main_events[main_events['position'] <= 3].groupby('rider').size().sort_values(ascending=False)

print("Top 15 riders by podiums (2023-2025):")
print(podiums.head(15))

# Compare wins vs podiums for top riders
top_riders = wins.head(10).index
comparison = pd.DataFrame({
    'Wins': wins[top_riders],
    'Podiums': podiums[top_riders]
})

comparison.plot(kind='bar', figsize=(12, 6))
plt.title('Wins vs Podiums - Top 10 Riders')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.legend(['Wins', 'Podiums'])
plt.tight_layout()
plt.show()

In [ ]:
# Average finishing position by rider (min 5 races)
rider_stats = main_events.groupby('rider').agg({
    'position': ['mean', 'median', 'std', 'count'],
    'race': 'nunique'
}).round(2)

rider_stats.columns = ['avg_position', 'median_position', 'std_position', 'total_races', 'unique_races']
rider_stats = rider_stats[rider_stats['total_races'] >= 5].sort_values('avg_position')

print("Top 15 riders by average finishing position (min 5 races):")
print(rider_stats.head(15))

## 4. Class-Specific Analysis

In [ ]:
# Top riders by class
for class_name in ['450sx', '250sxe', '250sxw']:
    class_data = main_events[main_events['class'] == class_name]
    class_wins = class_data[class_data['position'] == 1].groupby('rider').size().sort_values(ascending=False)
    
    print(f"\n{'='*70}")
    print(f"TOP 10 RIDERS - {class_name.upper()}")
    print(f"{'='*70}")
    print(class_wins.head(10))

## 5. Qualifying vs Main Event Correlation

In [ ]:
# Get qualifying and main event results for same race/rider
qualifying = df[df['event_type'] == 'qualifying'][['year', 'round', 'race', 'class', 'rider', 'position']].copy()
qualifying.columns = ['year', 'round', 'race', 'class', 'rider', 'qual_position']

main = main_events[['year', 'round', 'race', 'class', 'rider', 'position']].copy()
main.columns = ['year', 'round', 'race', 'class', 'rider', 'main_position']

# Merge
qual_vs_main = qualifying.merge(main, on=['year', 'round', 'race', 'class', 'rider'], how='inner')

print(f"Matched qualifying-main event pairs: {len(qual_vs_main):,}")

# Calculate correlation
correlation = qual_vs_main['qual_position'].corr(qual_vs_main['main_position'])
print(f"\nCorrelation between qualifying and main event position: {correlation:.3f}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
plt.scatter(qual_vs_main['qual_position'], qual_vs_main['main_position'], alpha=0.3)
plt.xlabel('Qualifying Position')
plt.ylabel('Main Event Position')
plt.title(f'Qualifying vs Main Event Position (correlation: {correlation:.3f})')
plt.plot([0, 40], [0, 40], 'r--', alpha=0.5, label='Perfect correlation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Feature Engineering Ideas

Based on the exploration, here are potential features for the model:

In [ ]:
print("""
POTENTIAL FEATURES FOR MODEL:

1. Historical Performance:
   - Average finishing position (last 3, 5, 10 races)
   - Win rate / podium rate
   - Consistency (std of positions)
   - Recent form (weighted by recency)

2. Current Race Context:
   - Qualifying position
   - Heat race position
   - LCQ participation (indicator of struggle)
   - Starting position

3. Track/Venue:
   - Performance at specific track
   - Track type (stadium characteristics)
   - Weather conditions (if available)

4. Rider Characteristics:
   - Bike manufacturer
   - Class (450 vs 250)
   - Experience (number of races)

5. Competition:
   - Field strength (based on historical ratings)
   - Head-to-head records
   - Relative performance to field

6. Temporal:
   - Round number (season progression)
   - Days since last race
   - Season trends
""")

## 7. Data Quality Check

In [ ]:
# Check for duplicate results
duplicates = main_events.duplicated(subset=['year', 'round', 'race', 'class', 'rider'], keep=False)
print(f"Duplicate main event results: {duplicates.sum()}")

if duplicates.sum() > 0:
    print("\nDuplicate examples:")
    print(main_events[duplicates].sort_values(['year', 'round', 'race', 'rider']).head(10))

# Check for missing positions
print(f"\nMissing positions: {main_events['position'].isnull().sum()}")

# Check position distribution
print("\nPosition distribution:")
print(main_events['position'].describe())

## 8. Next Steps

1. **Feature Engineering**: Create historical performance features
2. **Train/Test Split**: Split by time (e.g., 2023-2024 train, 2025 test)
3. **Baseline Model**: XGBoost Regressor for position prediction
4. **Ranking Model**: XGBoost Rank (LambdaRank) for ranking prediction
5. **Evaluation**: Compare models using ranking metrics (NDCG, MAP)
6. **RL Environment**: Design state/action/reward for fantasy game

In [ ]:
# Save summary statistics for reference
summary = {
    'total_records': len(df),
    'main_events': len(main_events),
    'unique_riders': df['rider'].nunique(),
    'unique_races': df['race'].nunique(),
    'years': sorted(df['year'].unique()),
    'classes': sorted(df['class'].unique()),
    'qual_main_correlation': correlation
}

print("\nSummary:")
for key, value in summary.items():
    print(f"  {key}: {value}")